# UCS420: Cognitive Computing — Lab Assignment 9 (Part B)
## Emotion-aware Cognitive Assistant using Facial Expression Detection
*Vision-based Cognitive System (Landing AI theme)*

The assistant **sees** the user (webcam or photo), **detects the face**, **recognises the facial expression** with a CNN trained on FER-2013, **reasons** about the user's emotional state over time (rules plus emotion history), and **responds** empathetically, adapting what it says to the user's mood.

```
 ┌────────────┐   ┌──────────────┐   ┌───────────────────┐   ┌──────────────────────┐   ┌──────────────┐
 │ Webcam /   │──▶│ Face         │──▶│ Expression CNN    │──▶│ Reasoning            │──▶│ Empathetic   │
 │ photo      │   │ detection    │   │ (7 emotions,      │   │ confidence, history, │   │ response (+  │
 └────────────┘   │ (Haar/OpenCV)│   │  FER-2013)        │   │ mood trend, rules    │   │ speech)      │
                  └──────────────┘   └───────────────────┘   └──────────────────────┘   └──────────────┘
```

**Dataset:** [FER-2013](https://www.kaggle.com/datasets/msambare/fer2013), 35,887 grayscale 48×48 face images in 7 classes (angry, disgust, fear, happy, neutral, sad, surprise), downloaded from Kaggle with `kagglehub`.

⚙️ **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Training takes about 15–25 minutes on a T4.

## 0. Setup

In [ ]:
!pip -q install kagglehub gTTS

In [ ]:
import os, glob, random, time, warnings
from collections import deque, Counter
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Download the FER-2013 Dataset

In [ ]:
import kagglehub
try:
    DATA_DIR = kagglehub.dataset_download("msambare/fer2013")
except Exception as e:
    print("Anonymous download failed:", e)
    print("→ Log in with your Kaggle API token (kaggle.com → Settings → Create New Token):")
    kagglehub.login()
    DATA_DIR = kagglehub.dataset_download("msambare/fer2013")

TRAIN_DIR, TEST_DIR = os.path.join(DATA_DIR, "train"), os.path.join(DATA_DIR, "test")
CLASSES = sorted(os.listdir(TRAIN_DIR))
print("Data:", DATA_DIR, "\nClasses:", CLASSES)

counts = pd.DataFrame({split: {c: len(os.listdir(os.path.join(d, c))) for c in CLASSES}
                       for split, d in [("train", TRAIN_DIR), ("test", TEST_DIR)]})
print(counts, "\nTotal:", counts.values.sum())

## 2. Exploratory Data Analysis
The dataset is **imbalanced**: *happy* has about 7,000 training images and *disgust* only about 400. Class weights (§4) compensate for this.

In [ ]:
counts.plot.bar(figsize=(9, 4), color=["#457b9d", "#e76f51"], title="Images per class")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(len(CLASSES), 8, figsize=(12, 11))
for i, c in enumerate(CLASSES):
    files = random.sample(os.listdir(os.path.join(TRAIN_DIR, c)), 8)
    for j, f in enumerate(files):
        axes[i, j].imshow(Image.open(os.path.join(TRAIN_DIR, c, f)), cmap="gray"); axes[i, j].axis("off")
    axes[i, 0].set_title(c, loc="left", fontsize=11)
plt.suptitle("FER-2013 samples"); plt.tight_layout(); plt.show()

## 3. Input Pipeline
85% / 15% train / validation split of the training folder; the official **test** folder is held out for final evaluation.

In [ ]:
IMG, BATCH = 48, 64
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.15, subset="both", seed=SEED, color_mode="grayscale",
    image_size=(IMG, IMG), batch_size=BATCH, class_names=CLASSES)
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, color_mode="grayscale", image_size=(IMG, IMG), batch_size=BATCH,
    shuffle=False, class_names=CLASSES)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(4000, seed=SEED).prefetch(AUTOTUNE)
val_ds, test_ds = val_ds.cache().prefetch(AUTOTUNE), test_ds.cache().prefetch(AUTOTUNE)

y_train = np.concatenate([y.numpy() for _, y in train_ds])
cw = compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y_train)
class_weight = {i: float(np.sqrt(w)) for i, w in enumerate(cw)}   # softened (sqrt) balanced weights
print("Class weights:", {CLASSES[i]: round(w, 2) for i, w in class_weight.items()})

## 4. Model: Convolutional Neural Network
- **On-the-fly augmentation:** horizontal flip, rotation, zoom, translation, contrast
- **4 convolutional blocks** (64 → 128 → 256 → 512 filters), each `Conv-BN-ReLU ×2 → MaxPool → Dropout`
- **Classifier head:** `Dense(256) → BN → ReLU → Dropout(0.5) → Softmax(7)`
- **Training:** Adam, class weights, EarlyStopping and ReduceLROnPlateau on validation accuracy

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.2),
], name="augmentation")

def conv_block(x, filters, drop):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                          kernel_regularizer=keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(drop)(x)

def build_model(n_classes=len(CLASSES)):
    inp = keras.Input((IMG, IMG, 1))
    x = augment(inp)
    x = layers.Rescaling(1 / 255.0)(x)
    for f, d in [(64, 0.2), (128, 0.25), (256, 0.3), (512, 0.35)]:
        x = conv_block(x, f, d)
    x = layers.Flatten()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inp, out, name="EmotionCNN")

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
EPOCHS = 60          # early stopping usually ends training at around 35–50 epochs
MODEL_PATH = "emotion_cnn.keras"

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True),
]
t0 = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    class_weight=class_weight, callbacks=callbacks, verbose=2)
print(f"Training time: {(time.time() - t0) / 60:.1f} min")
model = keras.models.load_model(MODEL_PATH)

In [ ]:
h = pd.DataFrame(history.history)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
h[["accuracy", "val_accuracy"]].plot(ax=ax[0], title="Accuracy"); ax[0].set_xlabel("epoch")
h[["loss", "val_loss"]].plot(ax=ax[1], title="Loss"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 5. Evaluation on the Held-out Test Set
Uses **test-time augmentation** (average of the prediction on the image and on its mirror image), which usually adds about 1% accuracy.

In [ ]:
X_test = np.concatenate([x.numpy() for x, _ in test_ds])
y_test = np.concatenate([y.numpy() for _, y in test_ds])

def predict_tta(x):
    p = model.predict(x, batch_size=256, verbose=0)
    p_flip = model.predict(x[:, :, ::-1, :], batch_size=256, verbose=0)
    return (p + p_flip) / 2

probs_test = predict_tta(X_test)
y_pred = probs_test.argmax(1)
print(f"Test accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1      : {f1_score(y_test, y_pred, average='macro'):.4f}\n")
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_test, y_pred, normalize="true")
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Purples", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("Normalised confusion matrix (test set)"); plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()

In [ ]:
idx = np.random.choice(len(X_test), 16, replace=False)
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(X_test[i].squeeze(), cmap="gray"); ax.axis("off")
    ok = y_pred[i] == y_test[i]
    ax.set_title(f"T:{CLASSES[y_test[i]]}\nP:{CLASSES[y_pred[i]]} {probs_test[i].max():.2f}",
                 color="green" if ok else "red", fontsize=9)
plt.suptitle("Sample test predictions (green = correct)"); plt.tight_layout(); plt.show()

## 6. Perception: Face Detection + Expression Recognition
OpenCV's Haar-cascade detector finds faces in a full photo. Each face is cropped with a small margin, converted to 48×48 grayscale (the same format as FER-2013) and passed to the CNN. Images that are already 48×48 face crops skip detection.

In [ ]:
FACE_DET = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def detect_faces(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = FACE_DET.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(48, 48))
    return sorted([tuple(map(int, f)) for f in faces], key=lambda f: -f[2] * f[3])   # largest first

def face_to_input(gray_face):
    face = cv2.resize(gray_face, (IMG, IMG), interpolation=cv2.INTER_AREA)
    return face.astype("float32")[None, :, :, None]

def analyse_image(img):
    """Returns (rgb image, list of faces: {box, probs})."""
    rgb = np.array(img.convert("RGB")) if isinstance(img, Image.Image) else img
    if rgb.ndim == 2:
        rgb = cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    if max(h, w) <= 64:                                   # already a face crop
        return rgb, [{"box": (0, 0, w, h), "probs": predict_tta(face_to_input(gray))[0]}]
    faces = []
    for (x, y, fw, fh) in detect_faces(rgb):
        m = int(0.1 * fw)
        crop = gray[max(0, y - m): y + fh + m, max(0, x - m): x + fw + m]
        faces.append({"box": (x, y, fw, fh), "probs": predict_tta(face_to_input(crop))[0]})
    return rgb, faces

## 7. Reasoning: The Emotion-aware Cognitive Layer

**Knowledge: emotion → valence → response strategy**

| Emotion | Valence | Assistant strategy |
|---|---|---|
| happy | positive | Mirror the positivity, encourage |
| surprise | neutral/positive | Check in: good or bad surprise? |
| neutral | neutral | Normal, task-focused help |
| sad | negative | Empathy, gentle support, offer something uplifting |
| angry | negative | Calm, de-escalate, suggest a pause and breathing |
| fear | negative | Reassure, break the problem into steps |
| disgust | negative | Acknowledge, offer to change topic/task |

**Rules**

| # | Rule | Condition | Action |
|---|---|---|---|
| R1 | No face | no face detected | Ask the user to face the camera / improve lighting |
| R2 | Multiple faces | > 1 face | Respond to the main (largest) face, report the others |
| R3 | **Low confidence** | top probability < 0.40 **or** top-2 margin < 0.10 | Don't assume; ask how the user feels and mention both candidates |
| R4 | Emotion-specific response | confident prediction | Use the strategy for that emotion |
| R5 | **Persistent negative mood** | current reading negative **and** ≥ 3 negative of the last 5 readings | Escalate support: suggest a break or talking to someone; share Tele-MANAS 14416 |
| R6 | Mood improvement | negative → positive transition | Acknowledge the improvement |
| R7 | Context adaptation | `mode="study"` | Tailor suggestions to studying (breaks, chunking, Pomodoro) |
| R8 | Stable / no change | same emotion as last time | Vary the wording so the assistant doesn't repeat itself |

In [ ]:
VALENCE = {"happy": "positive", "surprise": "neutral", "neutral": "neutral",
           "sad": "negative", "angry": "negative", "fear": "negative", "disgust": "negative"}
EMOJI = {"happy": "😊", "surprise": "😮", "neutral": "😐", "sad": "😢", "angry": "😠", "fear": "😨", "disgust": "🤢"}

RESPONSES = {
    "happy":    ["You look happy! Great energy, what's going well today?",
                 "That's a lovely smile! Let's make the most of this good mood.",
                 "You seem cheerful. Want to tackle something challenging while you're feeling good?"],
    "surprise": ["You look surprised! Was it a good surprise or an unexpected problem?",
                 "Something caught you off guard? Tell me what happened."],
    "neutral":  ["You seem calm and focused. How can I help you right now?",
                 "All steady. Ready when you are: what would you like to do next?"],
    "sad":      ["You look a little down. I'm here if you want to talk about it.",
                 "It seems like you might be feeling sad. Would something light help, like music, a short walk or a chat?",
                 "I'm sorry if today's been hard. Be gentle with yourself."],
    "angry":    ["You seem frustrated. Let's pause for a moment. Try a slow breath in for 4 and out for 6.",
                 "I can sense some frustration. Want to tell me what's bothering you? We can sort it out step by step."],
    "fear":     ["You look worried. It's okay. Let's break whatever it is into small, manageable steps.",
                 "Feeling anxious? You're not alone. What's the one thing worrying you most right now?"],
    "disgust":  ["Something seems to have put you off. Want to switch to a different task or topic?",
                 "That doesn't look pleasant! Shall we change what we're doing?"],
}
STUDY_TIPS = {
    "happy": "Good time for your hardest topic: start a 25-minute focus session.",
    "neutral": "Keep going: try a 25/5 Pomodoro cycle.",
    "surprise": "If you hit something confusing, write the question down and look it up in a moment.",
    "sad": "Take a 10-minute break: stretch, get water, step outside.",
    "angry": "Step away from the problem for 5 minutes; come back and re-read it fresh.",
    "fear": "Break the syllabus into tiny chunks and finish just one first. Small wins build confidence.",
    "disgust": "Switch subjects for a while to something you enjoy more.",
}
SUPPORT_MSG = ("💛 I've noticed you've seemed low or stressed several times in a row. Consider taking a proper "
               "break, or talking to a friend, family member or counsellor. If you're struggling, Tele-MANAS "
               "(14416, free, 24×7) offers confidential support.")

class EmotionAwareAssistant:
    def __init__(self, mode="general", conf_threshold=0.40, margin=0.10, window=5, speak=False):
        self.mode, self.conf_threshold, self.margin = mode, conf_threshold, margin
        self.history = deque(maxlen=window)
        self.log, self.speak_out = [], speak
        self.last_support_at = -99

    def _say(self, emotion):
        """Returns (response, repeated?). R8: vary the wording when the same emotion is seen again."""
        options = RESPONSES[emotion]
        seen = sum(l["emotion"] == emotion for l in self.log)
        prev = self.log[-1]["emotion"] if self.log else None
        return options[seen % len(options)], prev == emotion

    def respond_to_probs(self, probs, n_faces=1):
        order = np.argsort(probs)[::-1]
        top, second = CLASSES[order[0]], CLASSES[order[1]]
        conf, gap = float(probs[order[0]]), float(probs[order[0]] - probs[order[1]])
        rules, msg = [], []
        if n_faces > 1:
            rules.append("R2"); msg.append(f"👥 I can see {n_faces} people; I'll respond to the person closest to the camera.")
        if conf < self.conf_threshold or gap < self.margin:                                  # R3
            rules.append("R3")
            msg.append(f"🤔 I'm not sure how you're feeling: you might be {top} or {second}. How are you doing?")
            emotion = "uncertain"
        else:
            emotion = top
            prev_val = VALENCE[self.history[-1]] if self.history else None
            self.history.append(emotion)
            text, repeated = self._say(emotion)
            rules.append("R4"); msg.append(f"{EMOJI[emotion]} " + text)
            if repeated:
                rules.append("R8")
            if prev_val == "negative" and VALENCE[emotion] == "positive":                     # R6
                rules.append("R6"); msg.append("🌤️ Nice, you seem to be feeling better than before!")
            if self.mode == "study":                                                          # R7
                rules.append("R7"); msg.append("📚 Study tip: " + STUDY_TIPS[emotion])
            neg = sum(VALENCE[e] == "negative" for e in self.history)
            if VALENCE[emotion] == "negative" and neg >= 3 and len(self.log) - self.last_support_at >= 3:   # R5
                rules.append("R5"); msg.append(SUPPORT_MSG); self.last_support_at = len(self.log)
        entry = {"t": len(self.log), "emotion": emotion, "top": top, "confidence": round(conf, 3),
                 "margin": round(gap, 3), "rules": rules, "response": "\n".join(msg)}
        self.log.append(entry)
        return entry

    def respond(self, img, show=True):
        rgb, faces = analyse_image(img)
        if not faces:                                                                         # R1
            entry = {"t": len(self.log), "emotion": "no_face", "top": None, "confidence": None, "margin": None,
                     "rules": ["R1"], "response": "📷 I can't see a face. Please face the camera in good, even lighting."}
            self.log.append(entry)
        else:
            entry = self.respond_to_probs(faces[0]["probs"], n_faces=len(faces))
        if show:
            self.show(rgb, faces, entry)
        if self.speak_out:
            self.speak(entry["response"])
        return entry

    @staticmethod
    def show(rgb, faces, entry):
        vis = rgb.copy()
        for k, f in enumerate(faces):
            x, y, w, h = f["box"]; e = CLASSES[int(np.argmax(f["probs"]))]
            col = (0, 200, 0) if k == 0 else (200, 200, 0)
            cv2.rectangle(vis, (x, y), (x + w, y + h), col, max(1, vis.shape[1] // 300))
            cv2.putText(vis, f"{e} {f['probs'].max():.2f}", (x, max(12, y - 6)), cv2.FONT_HERSHEY_SIMPLEX,
                        max(0.35, vis.shape[1] / 1200), col, max(1, vis.shape[1] // 400))
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1.3]})
        ax[0].imshow(vis); ax[0].axis("off")
        if faces:
            p = faces[0]["probs"]
            ax[1].barh(CLASSES, p, color=["#2a9d8f" if c == CLASSES[int(np.argmax(p))] else "#ccc" for c in CLASSES])
            ax[1].set_xlim(0, 1); ax[1].set_title("Emotion probabilities (main face)")
        else:
            ax[1].axis("off")
        plt.tight_layout(); plt.show()
        print(f"[rules: {', '.join(entry['rules'])}]\n🤖 {entry['response']}\n")

    def mood_report(self):
        df = pd.DataFrame(self.log)
        if df.empty or not (~df["emotion"].isin(["no_face", "uncertain"])).any():
            print("No confident readings yet."); return
        df = df[~df["emotion"].isin(["no_face", "uncertain"])]
        val_score = df["emotion"].map(lambda e: {"positive": 1, "neutral": 0, "negative": -1}[VALENCE[e]])
        fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
        ax[0].plot(df["t"], val_score, marker="o", color="#6a4c93")
        ax[0].set_yticks([-1, 0, 1], ["negative", "neutral", "positive"]); ax[0].set_title("Mood trend")
        ax[0].set_xlabel("reading #")
        df["emotion"].value_counts().plot.bar(ax=ax[1], color="#457b9d"); ax[1].set_title("Emotion counts")
        plt.tight_layout(); plt.show()

    @staticmethod
    def speak(text):
        try:
            from gtts import gTTS
            from IPython.display import Audio, display
            import re
            gTTS(re.sub(r"[^\w\s.,!?'’:-]", "", text)).save("reply.mp3")
            display(Audio("reply.mp3", autoplay=True))
        except Exception as e:
            print("(speech unavailable:", e, ")")

## 8. Testing the Assistant
### 8a. A simulated session using unseen test-set faces
The sequence *happy → neutral → sad → fear → sad → angry → happy* shows the rules in action: emotion-specific responses (R4), persistent negative mood escalation (R5), mood improvement (R6) and low-confidence handling (R3).

In [ ]:
def pick_test_face(emotion, confident=True):
    """Pick a test image of the given class that the model classifies correctly and confidently."""
    c = CLASSES.index(emotion)
    cand = np.where((y_test == c) & (y_pred == c) & ((probs_test.max(1) > 0.6) if confident else True))[0]
    return X_test[np.random.choice(cand)].squeeze().astype(np.uint8)

assistant = EmotionAwareAssistant(mode="general")
for emo in ["happy", "neutral", "sad", "fear", "sad", "angry", "happy"]:
    print(f"━━━━━━━━ Input: a '{emo}' face ━━━━━━━━")
    assistant.respond(pick_test_face(emo))

In [ ]:
# Low-confidence case (R3): the test image on which the model is least certain
amb = int(np.argmin(probs_test.max(1)))
print("━━━━━━━━ Ambiguous face ━━━━━━━━")
assistant.respond(X_test[amb].squeeze().astype(np.uint8))

# No-face case (R1): a blank image
print("━━━━━━━━ No face ━━━━━━━━")
assistant.respond(np.full((240, 320, 3), 200, np.uint8))

assistant.mood_report()
pd.DataFrame(assistant.log)[["t", "emotion", "confidence", "margin", "rules"]]

In [ ]:
# Study-mode (R7): same faces, assistant adapts its advice to a student context
study_bot = EmotionAwareAssistant(mode="study")
for emo in ["fear", "neutral", "happy"]:
    print(f"━━━━━━━━ Study mode, '{emo}' face ━━━━━━━━")
    study_bot.respond(pick_test_face(emo))

### 8b. Upload your own photos
Full-size photos are supported: faces are detected automatically, and group photos trigger R2.

In [ ]:
try:
    from google.colab import files
    for name in files.upload():
        print(f"━━━━━━━━ {name} ━━━━━━━━")
        assistant.respond(Image.open(name))
except Exception as e:
    print("Upload works in Google Colab:", e)

### 8c. Live webcam session (Google Colab)
Click **Capture** each time; the assistant analyses your expression and replies. Try smiling, frowning or looking surprised. Press Enter for the next capture or type `q` to finish. Set `speak=True` to hear the replies.

In [ ]:
from IPython.display import display, Javascript
from base64 import b64decode

def take_photo(filename="photo.jpg", quality=0.8):
    from google.colab.output import eval_js
    js = Javascript("""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Capture';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    """)
    display(js)
    data = eval_js(f"takePhoto({quality})")
    with open(filename, "wb") as f:
        f.write(b64decode(data.split(",")[1]))
    return filename

def webcam_session(mode="general", speak=False):
    bot = EmotionAwareAssistant(mode=mode, speak=speak)
    print("🎥 Emotion-aware assistant: webcam session")
    while True:
        try:
            bot.respond(Image.open(take_photo()))
        except Exception as e:
            print("Webcam error (allow camera access in the browser):", e); break
        if input("Enter = next capture, q = quit: ").strip().lower() == "q":
            break
    bot.mood_report()
    return bot

session = webcam_session(mode="general", speak=False)

## 9. Save / Download the Model

In [ ]:
model.save(MODEL_PATH)
print("Saved", MODEL_PATH, f"({os.path.getsize(MODEL_PATH) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(MODEL_PATH)
except Exception:
    pass

## Conclusion
- **Perception:** OpenCV Haar-cascade face detection plus a 4-block CNN trained from scratch on FER-2013 (augmentation, batch norm, dropout, class weights, test-time augmentation). FER-2013 is a hard, noisy dataset: human accuracy is around 65 ± 5%, and models of this size typically reach **about 63–68%** test accuracy. *Happy* and *surprise* are recognised best; *fear* and *disgust* are hardest.
- **Cognition:** emotion → valence knowledge, a rolling emotion history and 8 rules turn raw predictions into context-aware, empathetic behaviour. It handles uncertainty explicitly (R3) instead of over-claiming, escalates support only when a negative mood **persists** (R5), and adapts to the context (study mode, R7).
- **Limitations & ethics:** a facial expression is not the same as an internal feeling; predictions vary with lighting, pose, age and ethnicity, and FER-2013 has known demographic biases. The assistant therefore phrases its readings tentatively ("you *seem*…"), never stores images, and should be used only with the user's consent.
- **Future work:** transfer learning (e.g. EfficientNet / ViT on FER+ or AffectNet), a DNN/MediaPipe face detector, and multimodal cues (voice tone, text sentiment).